# QUBO Portfolio Optimization - Google Colab Version
## Standalone notebook with QUBO optimization engines
### Tests: Multiple QUBO optimization methods for portfolio optimization

In [ ]:
# Install required packages
%pip install riskfolio-lib
%pip install dwave-ocean-sdk
%pip install dimod
%pip install yfinance
%pip install matplotlib
%pip install seaborn
%pip install pandas
%pip install numpy
%pip install scipy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time
import logging
import os
import warnings
import itertools
from datetime import datetime, timedelta
from functools import partial
from pathlib import Path
import pickle

# D-Wave imports
from dimod import BinaryQuadraticModel, ConstrainedQuadraticModel, Binary, quicksum
from dwave.samplers import SimulatedAnnealingSampler, TabuSampler, SteepestDescentSampler
from dwave.system import LeapHybridBQMSampler, LeapHybridCQMSampler, LeapHybridSampler
from dimod import ExactSolver

# Other optimization imports
import scipy.optimize as optimize
import riskfolio as rp

# Set random seed
seed = 12
np.random.seed(seed)

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("qubo_portfolio_logger")

# Suppress warnings
warnings.filterwarnings('ignore')

In [ ]:
# QUBO Optimization Engines from optimization_engines_v2

def portfolio_stats(weights, data):
    """Calculate portfolio statistics (return, volatility)"""
    weights = np.array(weights)
    returns = np.log(data) - np.log(data.shift(1))  # log return to minimize fp error
    port_return = np.sum(returns.mean() * weights) 
    port_vol = np.sqrt(np.dot(weights.T, np.dot(returns.cov(), weights)))
    return port_return, port_vol


def qubo_fitness_function(weights, data, risk_level=0.5):
    """
    QUBO fitness function with fixed risk level.
    
    Maximizes: return - risk_level * variance
    Fixed risk level of 0.5 for consistent comparison across QUBO optimizers.
    
    Args:
        weights: Portfolio weights
        data: Stock price data
        risk_level: Risk aversion parameter (fixed at 0.5)
    
    Returns:
        Objective value (return - risk * variance) for maximization
    """
    port_return, port_vol = portfolio_stats(weights, data)
    portfolio_variance = port_vol ** 2
    objective = port_return - risk_level * portfolio_variance
    return objective


def equal_weights_baseline(data):
    """
    Equal weights baseline portfolio.
    
    Args:
        data: Stock price DataFrame
        
    Returns:
        weights: Equal weight allocation
    """
    n_assets = len(data.columns)
    return np.full(n_assets, 1.0 / n_assets)


def genetic_algorithm_qubo(data, population_size=500, num_generations=100, mutation_rate=0.1, elitism=0.1):
    """
    Genetic Algorithm for QUBO portfolio optimization.
    
    Uses fixed risk level of 0.5 in the objective function.
    
    Args:
        data: Stock price DataFrame
        population_size: Number of individuals in each generation
        num_generations: Number of generations to evolve
        mutation_rate: Probability of mutation for each gene
        elitism: Fraction of best individuals to preserve
        
    Returns:
        weights: Optimized portfolio weights
    """
    n_assets = len(data.columns)
    n_elite = int(population_size * elitism)
    
    # Initialize population with normalized random weights
    population = []
    for _ in range(population_size):
        weights = np.random.rand(n_assets)
        weights = weights / weights.sum()  # Normalize to sum to 1
        population.append(weights)
    
    def fitness_function(weights):
        return qubo_fitness_function(weights, data, risk_level=0.5)
    
    for generation in range(num_generations):
        # Calculate fitness for all individuals
        fitness_scores = [fitness_function(individual) for individual in population]
        
        # Sort population by fitness (descending - higher is better)
        sorted_indices = np.argsort(fitness_scores)[::-1]
        population = [population[i] for i in sorted_indices]
        fitness_scores = [fitness_scores[i] for i in sorted_indices]
        
        # Elite selection
        new_population = population[:n_elite].copy()
        
        # Fill rest of population through crossover and mutation
        while len(new_population) < population_size:
            # Tournament selection
            parent1_idx = np.random.choice(min(50, len(population)))
            parent2_idx = np.random.choice(min(50, len(population)))
            parent1 = population[parent1_idx]
            parent2 = population[parent2_idx]
            
            # Uniform crossover
            child = np.where(np.random.rand(n_assets) < 0.5, parent1, parent2)
            
            # Mutation
            if np.random.rand() < mutation_rate:
                mutation_strength = 0.1
                noise = np.random.normal(0, mutation_strength, n_assets)
                child = child + noise
                child = np.abs(child)  # Ensure non-negative
            
            # Normalize weights to sum to 1
            child = child / child.sum() if child.sum() > 0 else np.full(n_assets, 1.0/n_assets)
            new_population.append(child)
        
        population = new_population
    
    # Return best individual
    final_fitness = [fitness_function(individual) for individual in population]
    best_idx = np.argmax(final_fitness)
    return population[best_idx]


def scipy_slsqp_qubo(data):
    """
    SciPy SLSQP optimizer for QUBO portfolio optimization.
    
    Uses fixed risk level of 0.5 in the objective function.
    
    Args:
        data: Stock price DataFrame
        
    Returns:
        weights: Optimized portfolio weights
    """
    n_assets = len(data.columns)
    
    def objective_function(weights):
        # Minimize negative objective (maximize objective)
        return -qubo_fitness_function(weights, data, risk_level=0.5)
    
    # Constraints: weights sum to 1, all weights >= 0
    constraints = [
        {'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0},  # Budget constraint
    ]
    
    bounds = [(0.001, 1.0) for _ in range(n_assets)]  # Min investment constraint
    
    # Initial guess (equal weights)
    x0 = np.full(n_assets, 1.0 / n_assets)
    
    try:
        result = optimize.minimize(
            objective_function,
            x0,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints,
            options={'maxiter': 1000}
        )
        
        if result.success:
            return result.x
        else:
            logger.warning(f"SciPy optimization failed: {result.message}")
            return x0
            
    except Exception as e:
        logger.error(f"SciPy SLSQP optimization error: {e}")
        return x0


def riskfolio_qubo(data):
    """
    Riskfolio-lib optimizer for QUBO portfolio optimization.
    
    Uses mean-variance optimization with fixed risk aversion.
    
    Args:
        data: Stock price DataFrame
        
    Returns:
        weights: Optimized portfolio weights
    """
    try:
        # Calculate returns
        returns = data.pct_change().dropna()
        
        # Create Portfolio object
        port = rp.Portfolio(returns=returns)
        
        # Calculate risk and return parameters
        port.assets_stats(method_mu='hist', method_cov='hist')
        
        # Fixed risk aversion parameter (equivalent to risk_level=0.5)
        # In Riskfolio, risk aversion (rm) controls the risk-return tradeoff
        risk_aversion = 1.0  # Corresponds roughly to risk_level=0.5
        
        # Mean-variance optimization
        weights = port.optimization(
            model='Classic',
            rm='MV',  # Mean Variance
            obj='Utility',  # Utility maximization (return - risk_aversion * risk)
            rf=0.0,  # Risk-free rate
            l=risk_aversion,  # Risk aversion parameter
            hist=True
        )
        
        if weights is not None and len(weights) > 0:
            return weights.values.flatten()
        else:
            logger.warning("Riskfolio optimization returned empty result")
            n_assets = len(data.columns)
            return np.full(n_assets, 1.0 / n_assets)
            
    except Exception as e:
        logger.error(f"Riskfolio optimization error: {e}")
        n_assets = len(data.columns)
        return np.full(n_assets, 1.0 / n_assets)


def dwave_bqm_qubo(data, budget=1.0, min_investment=0.001):
    """
    D-Wave BQM (Binary Quadratic Model) for QUBO portfolio optimization.
    
    Uses fixed risk level of 0.5 in the QUBO formulation.
    
    Args:
        data: Stock price DataFrame
        budget: Total budget (default 1.0 for normalized weights)
        min_investment: Minimum investment per asset
        
    Returns:
        weights: Optimized portfolio weights
    """
    try:
        # Prepare data
        returns = np.log(data) - np.log(data.shift(1))
        expected_returns = returns.mean().values
        covariance_matrix = returns.cov().values
        n_assets = len(expected_returns)
        
        # QUBO formulation parameters
        risk_level = 0.5
        precision = 100  # Scale factor for integer weights
        
        # Create BQM
        bqm = BinaryQuadraticModel({}, {}, 0.0, 'BINARY')
        
        # Create binary variables for each possible investment level per asset
        # Each asset can have investment levels: 0, min_investment, 2*min_investment, ..., budget
        max_investment_units = int(budget / min_investment)
        
        # Variables: x[asset][level] = 1 if asset has investment level
        variables = {}
        for i in range(n_assets):
            for level in range(max_investment_units + 1):
                var_name = f'x_{i}_{level}'
                variables[(i, level)] = var_name
                bqm.add_variable(var_name, 0.0)
        
        # Constraint: exactly one investment level per asset
        for i in range(n_assets):
            asset_vars = [variables[(i, level)] for level in range(max_investment_units + 1)]
            # Add penalty for not selecting exactly one level
            penalty = 1000.0
            for j, var1 in enumerate(asset_vars):
                for k, var2 in enumerate(asset_vars):
                    if j == k:
                        bqm.add_variable(var1, -penalty)  # Reward selecting one
                    else:
                        bqm.add_interaction(var1, var2, penalty)  # Penalize selecting multiple
        
        # Budget constraint
        budget_penalty = 1000.0
        total_investment = 0
        for i in range(n_assets):
            for level in range(1, max_investment_units + 1):  # Skip level 0 (no investment)
                investment_amount = level * min_investment
                var_name = variables[(i, level)]
                total_investment += investment_amount
                # Penalty for exceeding budget
                if total_investment > budget:
                    bqm.add_variable(var_name, budget_penalty * (total_investment - budget))
        
        # Objective: maximize return - risk_level * variance
        for i in range(n_assets):
            for level in range(1, max_investment_units + 1):
                weight = (level * min_investment) / budget
                var_name = variables[(i, level)]
                # Add return component (negative because BQM minimizes)
                bqm.add_variable(var_name, -expected_returns[i] * weight * precision)
                
                # Add risk component (variance)
                for j in range(n_assets):
                    for level2 in range(1, max_investment_units + 1):
                        weight2 = (level2 * min_investment) / budget
                        var_name2 = variables[(j, level2)]
                        risk_term = risk_level * covariance_matrix[i][j] * weight * weight2 * precision
                        if i == j:
                            bqm.add_variable(var_name, risk_term)
                        else:
                            bqm.add_interaction(var_name, var_name2, risk_term / 2)
        
        # Solve with D-Wave
        try:
            sampler = LeapHybridBQMSampler()
            sampleset = sampler.sample(bqm, label="QUBO_Portfolio")
            
            if len(sampleset) > 0:
                best_sample = sampleset.first.sample
                
                # Extract weights from solution
                weights = np.zeros(n_assets)
                for i in range(n_assets):
                    for level in range(max_investment_units + 1):
                        var_name = variables[(i, level)]
                        if best_sample.get(var_name, 0) == 1:
                            weights[i] = (level * min_investment) / budget
                            break
                
                # Normalize weights
                if weights.sum() > 0:
                    weights = weights / weights.sum()
                    return weights
        
        except Exception as e:
            logger.warning(f"D-Wave BQM solver error: {e}")
        
        # Fallback to equal weights
        logger.warning("D-Wave BQM optimization failed. Using equal weights.")
        return np.full(n_assets, 1.0 / n_assets)
        
    except Exception as e:
        logger.error(f"D-Wave BQM setup error: {e}")
        n_assets = len(data.columns)
        return np.full(n_assets, 1.0 / n_assets)


def dwave_cqm_qubo(data, budget=1.0, min_investment=0.001):
    """
    D-Wave CQM (Constrained Quadratic Model) for QUBO portfolio optimization.
    
    Uses fixed risk level of 0.5 in the objective function.
    
    Args:
        data: Stock price DataFrame
        budget: Total budget (default 1.0 for normalized weights)
        min_investment: Minimum investment per asset
        
    Returns:
        weights: Optimized portfolio weights
    """
    try:
        # Prepare data
        returns = np.log(data) - np.log(data.shift(1))
        expected_returns = returns.mean().values
        covariance_matrix = returns.cov().values
        n_assets = len(expected_returns)
        
        # Create CQM
        cqm = ConstrainedQuadraticModel()
        
        # Binary variables: whether to invest in each asset
        invest_binary = [Binary(f'invest_{i}') for i in range(n_assets)]
        
        # Integer variables: investment amounts (scaled by 1000 for precision)
        scale_factor = 1000
        max_investment = int(budget * scale_factor)
        investments = []
        for i in range(n_assets):
            var_name = f'investment_{i}'
            investments.append(var_name)
            cqm.add_variable('INTEGER', var_name, lower_bound=0, upper_bound=max_investment)
        
        # Budget constraint
        budget_constraint = quicksum(investments) <= int(budget * scale_factor)
        cqm.add_constraint(budget_constraint, label='budget')
        
        # Minimum investment constraints
        min_scaled_investment = int(min_investment * scale_factor)
        for i in range(n_assets):
            # If investing, must invest at least min_investment
            min_constraint = investments[i] >= min_scaled_investment * invest_binary[i]
            cqm.add_constraint(min_constraint, label=f'min_investment_{i}')
            
            # If not investing, investment is 0
            max_constraint = investments[i] <= max_investment * invest_binary[i]
            cqm.add_constraint(max_constraint, label=f'max_investment_{i}')
        
        # Force investment in all assets (every asset gets allocated)
        for i in range(n_assets):
            cqm.add_constraint(invest_binary[i] == 1, label=f'force_invest_{i}')
        
        # QUBO Objective: maximize return - risk_level * variance
        risk_level = 0.5
        objective = 0
        
        # Return component
        total_investment = quicksum(investments)
        for i in range(n_assets):
            weight = investments[i] / total_investment
            objective += expected_returns[i] * weight
        
        # Risk component (variance)
        for i in range(n_assets):
            for j in range(n_assets):
                weight_i = investments[i] / total_investment
                weight_j = investments[j] / total_investment
                objective -= risk_level * covariance_matrix[i][j] * weight_i * weight_j
        
        # Set objective (CQM minimizes, so negate for maximization)
        cqm.set_objective(-objective)
        
        # Solve with D-Wave
        try:
            sampler = LeapHybridCQMSampler()
            sampleset = sampler.sample_cqm(cqm, label="QUBO_CQM_Portfolio")
            
            if len(sampleset) > 0:
                best_sample = sampleset.first.sample
                
                # Extract investment amounts
                investment_amounts = np.zeros(n_assets)
                for i in range(n_assets):
                    investment_amounts[i] = best_sample.get(f'investment_{i}', 0) / scale_factor
                
                # Convert to weights
                total = investment_amounts.sum()
                if total > 0:
                    weights = investment_amounts / total
                    return weights
        
        except Exception as e:
            logger.warning(f"D-Wave CQM solver error: {e}")
        
        # Fallback to equal weights
        logger.warning("D-Wave CQM optimization failed. Using equal weights.")
        return np.full(n_assets, 1.0 / n_assets)
        
    except Exception as e:
        logger.error(f"D-Wave CQM setup error: {e}")
        n_assets = len(data.columns)
        return np.full(n_assets, 1.0 / n_assets)

In [ ]:
# Data generation and utilities

def generate_synthetic_stock_data(n_stocks, n_days, start_price=100, volatility=0.02, trend=0.001):
    """
    Generate synthetic stock price data.
    
    Args:
        n_stocks: Number of stocks
        n_days: Number of days
        start_price: Starting price for all stocks
        volatility: Daily volatility
        trend: Daily trend
    
    Returns:
        DataFrame with synthetic stock prices
    """
    np.random.seed(seed)
    
    # Generate correlated returns
    correlation_matrix = np.random.rand(n_stocks, n_stocks)
    correlation_matrix = (correlation_matrix + correlation_matrix.T) / 2
    np.fill_diagonal(correlation_matrix, 1.0)
    
    # Make it positive definite
    eigenvals, eigenvecs = np.linalg.eigh(correlation_matrix)
    eigenvals = np.maximum(eigenvals, 0.01)
    correlation_matrix = eigenvecs @ np.diag(eigenvals) @ eigenvecs.T
    
    # Generate returns
    returns = np.random.multivariate_normal(
        mean=np.full(n_stocks, trend),
        cov=correlation_matrix * (volatility ** 2),
        size=n_days
    )
    
    # Convert to prices
    prices = np.zeros((n_days, n_stocks))
    prices[0] = start_price
    
    for day in range(1, n_days):
        prices[day] = prices[day-1] * (1 + returns[day])
    
    # Create DataFrame
    stock_names = [f'Stock_{i+1}' for i in range(n_stocks)]
    dates = pd.date_range(start='2020-01-01', periods=n_days, freq='D')
    
    return pd.DataFrame(prices, index=dates, columns=stock_names)


def evaluate_portfolio(weights, data, method_name):
    """
    Evaluate portfolio performance.
    
    Args:
        weights: Portfolio weights
        data: Stock price data
        method_name: Name of optimization method
    
    Returns:
        Dictionary with performance metrics
    """
    port_return, port_vol = portfolio_stats(weights, data)
    sharpe_ratio = port_return / port_vol if port_vol > 0 else 0
    qubo_objective = qubo_fitness_function(weights, data, risk_level=0.5)
    
    return {
        'method': method_name,
        'return': port_return,
        'volatility': port_vol,
        'sharpe_ratio': sharpe_ratio,
        'qubo_objective': qubo_objective,
        'weights': weights
    }

In [ ]:
# Test configurations

test_configs = [
    {'n_stocks': 5, 'n_days': 252, 'name': '5 Stocks'},
    {'n_stocks': 10, 'n_days': 252, 'name': '10 Stocks'},
    {'n_stocks': 20, 'n_days': 252, 'name': '20 Stocks'}
]

# QUBO optimization methods to test
qubo_methods = {
    'Equal Weights': equal_weights_baseline,
    'SciPy SLSQP': scipy_slsqp_qubo,
    'Riskfolio': riskfolio_qubo,
    'Genetic Algorithm': genetic_algorithm_qubo,
    'D-Wave BQM': dwave_bqm_qubo,
    'D-Wave CQM': dwave_cqm_qubo
}

print("Test configurations:")
for config in test_configs:
    print(f"  - {config['name']}: {config['n_stocks']} stocks, {config['n_days']} days")

print(f"\nQUBO optimization methods: {list(qubo_methods.keys())}")

In [ ]:
# Run QUBO optimization experiments

results = []

for config in test_configs:
    print(f"\n=== Testing {config['name']} ===")
    
    # Generate synthetic data
    data = generate_synthetic_stock_data(
        n_stocks=config['n_stocks'], 
        n_days=config['n_days']
    )
    
    print(f"Generated data shape: {data.shape}")
    
    # Test each optimization method
    for method_name, optimizer_func in qubo_methods.items():
        print(f"\nTesting {method_name}...")
        
        try:
            start_time = time.time()
            
            # Run optimization
            if method_name == 'Genetic Algorithm':
                weights = optimizer_func(data, population_size=100, num_generations=50)
            else:
                weights = optimizer_func(data)
            
            end_time = time.time()
            optimization_time = end_time - start_time
            
            # Evaluate portfolio
            result = evaluate_portfolio(weights, data, method_name)
            result['config'] = config['name']
            result['n_stocks'] = config['n_stocks']
            result['optimization_time'] = optimization_time
            
            results.append(result)
            
            print(f"  Return: {result['return']:.4f}")
            print(f"  Volatility: {result['volatility']:.4f}")
            print(f"  Sharpe Ratio: {result['sharpe_ratio']:.4f}")
            print(f"  QUBO Objective: {result['qubo_objective']:.4f}")
            print(f"  Time: {optimization_time:.2f}s")
            
        except Exception as e:
            print(f"  Error: {str(e)}")
            logger.error(f"Error testing {method_name} on {config['name']}: {e}")

print(f"\nCompleted {len(results)} optimization runs.")

In [ ]:
# Analyze and visualize results

if results:
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    
    print("=== QUBO Portfolio Optimization Results ===")
    print("\nSummary Statistics:")
    print(results_df[['method', 'config', 'return', 'volatility', 'sharpe_ratio', 'qubo_objective', 'optimization_time']].round(4))
    
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. QUBO Objective comparison
    sns.barplot(data=results_df, x='method', y='qubo_objective', hue='config', ax=axes[0,0])
    axes[0,0].set_title('QUBO Objective Values by Method')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # 2. Return vs Risk scatter plot
    for config_name in results_df['config'].unique():
        config_data = results_df[results_df['config'] == config_name]
        axes[0,1].scatter(config_data['volatility'], config_data['return'], 
                         label=config_name, alpha=0.7, s=100)
        
        # Add method labels
        for _, row in config_data.iterrows():
            axes[0,1].annotate(row['method'][:3], 
                             (row['volatility'], row['return']), 
                             xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    axes[0,1].set_xlabel('Volatility (Risk)')
    axes[0,1].set_ylabel('Return')
    axes[0,1].set_title('Risk-Return Profile')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Sharpe Ratio comparison
    sns.barplot(data=results_df, x='method', y='sharpe_ratio', hue='config', ax=axes[1,0])
    axes[1,0].set_title('Sharpe Ratios by Method')
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # 4. Optimization time comparison
    sns.barplot(data=results_df, x='method', y='optimization_time', hue='config', ax=axes[1,1])
    axes[1,1].set_title('Optimization Time (seconds)')
    axes[1,1].tick_params(axis='x', rotation=45)
    axes[1,1].set_yscale('log')  # Log scale for time
    
    plt.tight_layout()
    plt.show()
    
    # Best performers analysis
    print("\n=== Best Performers ===")
    
    for config_name in results_df['config'].unique():
        config_results = results_df[results_df['config'] == config_name]
        
        best_qubo = config_results.loc[config_results['qubo_objective'].idxmax()]
        best_sharpe = config_results.loc[config_results['sharpe_ratio'].idxmax()]
        fastest = config_results.loc[config_results['optimization_time'].idxmin()]
        
        print(f"\n{config_name}:")
        print(f"  Best QUBO Objective: {best_qubo['method']} ({best_qubo['qubo_objective']:.4f})")
        print(f"  Best Sharpe Ratio: {best_sharpe['method']} ({best_sharpe['sharpe_ratio']:.4f})")
        print(f"  Fastest: {fastest['method']} ({fastest['optimization_time']:.2f}s)")
    
    # Weight distribution analysis for best QUBO performers
    print("\n=== Portfolio Weights for Best QUBO Performers ===")
    
    for config_name in results_df['config'].unique():
        config_results = results_df[results_df['config'] == config_name]
        best_qubo = config_results.loc[config_results['qubo_objective'].idxmax()]
        
        print(f"\n{config_name} - {best_qubo['method']}:")
        weights = best_qubo['weights']
        
        # Show top 5 weights
        sorted_indices = np.argsort(weights)[::-1][:5]
        for i, idx in enumerate(sorted_indices):
            print(f"  Asset {idx+1}: {weights[idx]:.3f}")
        
        print(f"  Weight concentration (Gini): {np.sum(np.abs(np.diff(np.sort(weights))))/(2*len(weights)*np.mean(weights)):.3f}")

else:
    print("No results to analyze. Check for errors in optimization runs.")

In [ ]:
# Save results for further analysis

if results:
    # Save to CSV
    results_df_save = results_df.drop('weights', axis=1)  # Remove weights array for CSV
    results_df_save.to_csv('qubo_portfolio_results.csv', index=False)
    print("Results saved to 'qubo_portfolio_results.csv'")
    
    # Save full results with weights to pickle
    with open('qubo_portfolio_results.pkl', 'wb') as f:
        pickle.dump(results, f)
    print("Full results with weights saved to 'qubo_portfolio_results.pkl'")
    
    # Summary statistics
    print("\n=== Final Summary ===")
    print(f"Total optimization runs: {len(results)}")
    print(f"Configurations tested: {results_df['config'].nunique()}")
    print(f"Methods tested: {results_df['method'].nunique()}")
    print(f"Average QUBO objective: {results_df['qubo_objective'].mean():.4f}")
    print(f"Average Sharpe ratio: {results_df['sharpe_ratio'].mean():.4f}")
    print(f"Average optimization time: {results_df['optimization_time'].mean():.2f}s")
else:
    print("No results to save.")